# CNN-LSTM Hybrid Model

In [ ]:
# Install dependencies
!pip install mediapipe==0.10.14

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import math
import random
import numpy as np
import mediapipe as mp
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
ZIP_PATH = '/content/drive/MyDrive/hf_asl_videos.zip'
VIDEO_ROOT = '/content/hf_asl_videos'

if not os.path.exists(VIDEO_ROOT):
    !unzip -q {ZIP_PATH} -d /content/

words = sorted([d for d in os.listdir(VIDEO_ROOT) if os.path.isdir(os.path.join(VIDEO_ROOT, d))])
print(f'Classes ({len(words)}): {words}')

In [ ]:
# video level split
random.seed(42)

video_splits = {'train': [], 'val': [], 'test': []}
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1

for word in words:
    word_dir = os.path.join(VIDEO_ROOT, word)
    videos = [f for f in os.listdir(word_dir) if f.endswith('.mp4')]
    random.shuffle(videos)

    n = len(videos)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)

    for v in videos[:n_train]:
        video_splits['train'].append((word, os.path.join(word_dir, v)))
    for v in videos[n_train:n_train+n_val]:
        video_splits['val'].append((word, os.path.join(word_dir, v)))
    for v in videos[n_train+n_val:]:
        video_splits['test'].append((word, os.path.join(word_dir, v)))

print(f"Train: {len(video_splits['train'])}")
print(f"Val: {len(video_splits['val'])}")
print(f"Test: {len(video_splits['test'])}")

## Hand Cropping Pipeline

In [ ]:
mp_hands = mp.solutions.hands

def crop_hand_from_frame(frame, hand_landmarks, padding=20):
    """ crops the image around hand boundary """
    h, w, _ = frame.shape
    x_min, y_min = w, h
    x_max, y_max = 0, 0

    for lm in hand_landmarks.landmark:
        x, y = int(lm.x * w), int(lm.y * h)
        x_min = min(x_min, x)
        y_min = min(y_min, y)
        x_max = max(x_max, x)
        y_max = max(y_max, y)

    # pad
    x_min = max(0, x_min - padding)
    y_min = max(0, y_min - padding)
    x_max = min(w, x_max + padding)
    y_max = min(h, y_max + padding)

    # extract crop
    crop = frame[y_min:y_max, x_min:x_max]

    # make sure crop is big enough
    if crop.shape[0] < 10 or crop.shape[1] < 10:
        return None

    return crop

def process_video_for_crops(video_path, save_dir, frames_per_video=10):
    """ extracts N cropped hand images from a video """
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frames.append(frame)
    cap.release()

    if len(frames) < frames_per_video:
        return 0

    # evenly sample frames
    indices = np.linspace(0, len(frames)-1, frames_per_video, dtype=int)
    saved_count = 0

    with mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5) as hands:
        for i, idx in enumerate(indices):
            frame = frames[idx]
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(rgb_frame)

            if results.multi_hand_landmarks:
                # crop around first detected hand
                crop = crop_hand_from_frame(frame, results.multi_hand_landmarks[0])
                if crop is not None:
                    crop = cv2.resize(crop, (128, 128))
                    save_path = os.path.join(save_dir, f'frame_{i}.jpg')
                    cv2.imwrite(save_path, crop)
                    saved_count += 1

    return saved_count

In [ ]:
CROPPED_ROOT = '/content/cropped_hands'
os.makedirs(CROPPED_ROOT, exist_ok=True)

dataset_crops = [] # (image_path, label_idx)

for split in ['train', 'val']:
    for word, vid_path in tqdm(video_splits[split], desc=f'Processing {split}'):
        vid_name = os.path.basename(vid_path).replace('.mp4', '')
        save_dir = os.path.join(CROPPED_ROOT, split, word, vid_name)
        os.makedirs(save_dir, exist_ok=True)

        count = process_video_for_crops(vid_path, save_dir)

        if count > 0:
            label = words.index(word)
            for img_name in os.listdir(save_dir):
                dataset_crops.append((os.path.join(save_dir, img_name), label, split))

print(f'Total cropped images: {len(dataset_crops)}')

## Train CNN on Cropped Hands Images

In [ ]:
class HandCropDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, _ = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

train_samples = [x for x in dataset_crops if x[2] == 'train']
val_samples = [x for x in dataset_crops if x[2] == 'val']

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = HandCropDataset(train_samples, transform)
val_ds = HandCropDataset(val_samples, transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

print(f'CNN Train Images: {len(train_ds)}')
print(f'CNN Val Images: {len(val_ds)}')

In [ ]:
# CNN model
cnn_model = resnet18(weights=ResNet18_Weights.DEFAULT)
cnn_model.fc = nn.Linear(cnn_model.fc.in_features, len(words))
cnn_model = cnn_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=1e-4)

best_acc = 0

for epoch in range(10):
    cnn_model.train()
    for imgs, labels in tqdm(train_loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = cnn_model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()

    # validation
    cnn_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = cnn_model(imgs)
            correct += (out.argmax(1) == labels).sum().item()
            total += len(labels)

    acc = correct / total
    print(f'Epoch {epoch+1}: Val Acc = {acc:.4f}')
    if acc > best_acc:
        best_acc = acc
        torch.save(cnn_model.state_dict(), 'best_hand_cnn.pth')


## Extract Features and Train LSTM

In [ ]:
# feature extraction
cnn_model.load_state_dict(torch.load('best_hand_cnn.pth'))
cnn_model.fc = nn.Identity()  # remove classification layer to get 512-dim features
cnn_model.eval()

FEATURES_DIR = '/content/hand_features'
os.makedirs(FEATURES_DIR, exist_ok=True)

def extract_features(video_path, split_name):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frames.append(frame)
    cap.release()

    if len(frames) < 10: return None

    # 30 frames max per video
    indices = np.linspace(0, len(frames)-1, 30, dtype=int)

    features = []

    with mp_hands.Hands(static_image_mode=True, max_num_hands=1) as hands:
        for idx in indices:
            frame = frames[idx]
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(rgb)

            crop = None
            if results.multi_hand_landmarks:
                crop = crop_hand_from_frame(frame, results.multi_hand_landmarks[0])

            if crop is None:
                crop = np.zeros((128, 128, 3), dtype=np.uint8)
            else:
                crop = cv2.resize(crop, (128, 128))

            #  convert to tensor
            tensor = transform(Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))).unsqueeze(0).to(device)

            with torch.no_grad():
                feat = cnn_model(tensor)
                features.append(feat.cpu().numpy())

    return np.vstack(features)  # (30, 512)

all_sequences = []

for split in ['train', 'val', 'test']:
    for word, vid_path in tqdm(video_splits[split], desc=f'Extracting {split}'):
        feats = extract_features(vid_path, split)
        if feats is not None:
            label = words.index(word)
            all_sequences.append((feats, label, split))

print(f'Extracted sequences: {len(all_sequences)}')

In [ ]:
# LSTM Training
class SeqDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        return torch.from_numpy(self.data[idx][0]).float(), self.data[idx][1]

train_seqs = [x for x in all_sequences if x[2] == 'train']
val_seqs = [x for x in all_sequences if x[2] == 'val']
test_seqs = [x for x in all_sequences if x[2] == 'test']

train_loader = DataLoader(SeqDataset(train_seqs), batch_size=32, shuffle=True)
val_loader = DataLoader(SeqDataset(val_seqs), batch_size=32)
test_loader = DataLoader(SeqDataset(test_seqs), batch_size=32)

class LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim*2, num_classes)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

lstm_model = LSTM(512, 128, len(words)).to(device)
optimizer = optim.Adam(lstm_model.parameters(), lr=1e-3)

for epoch in range(30):
    lstm_model.train()
    for feats, labels in train_loader:
        feats, labels = feats.to(device), labels.to(device)
        optimizer.zero_grad()
        out = lstm_model(feats)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()

    lstm_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for feats, labels in val_loader:
            feats, labels = feats.to(device), labels.to(device)
            out = lstm_model(feats)
            correct += (out.argmax(1) == labels).sum().item()
            total += len(labels)
    print(f'Epoch {epoch+1}: Val Acc = {correct/total:.4f}')

# test
lstm_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for feats, labels in test_loader:
        feats, labels = feats.to(device), labels.to(device)
        out = lstm_model(feats)
        correct += (out.argmax(1) == labels).sum().item()
        total += len(labels)

print(f'Test Accuracy: {correct/total:.4f}')

In [ ]:
# save models
SAVE_DIR = '/content/drive/MyDrive/LearningASL_models'
os.makedirs(SAVE_DIR, exist_ok=True)
torch.save(cnn_model.state_dict(), os.path.join(SAVE_DIR, 'best_hand_cnn.pth'))
torch.save(lstm_model.state_dict(), os.path.join(SAVE_DIR, 'best_hand_lstm.pth'))